# Session 3 — TenderScope AI MCP demonstration
Run locally in WSL (not a hosted notebook). The MCP client launches `server.py` using stdio.


In [ ]:
from pathlib import Path
import sys
from mcp import Client, StdioServerParameters

PROJECT_ROOT = Path.home() / "tender_scope_ai"
SERVER = PROJECT_ROOT / "server.py"
PYTHON = PROJECT_ROOT / ".venv" / "bin" / "python"
assert SERVER.is_file(), f"Copy server.py to {SERVER}"
assert PYTHON.is_file(), f"Cannot find the project virtualenv at {PYTHON}"
print("Notebook interpreter:", sys.executable)
print("MCP server file:", SERVER)

params = StdioServerParameters(command=str(PYTHON), args=[str(SERVER)])


In [ ]:
async def run_s3_mcp_demo():
    async with Client(params) as client:
        listed = await client.list_tools()
        print("Discovered tools:", [tool.name for tool in listed.tools])
        assert [tool.name for tool in listed.tools] == ["rag_search_documents"]
        print("Read only:", listed.tools[0].annotations.read_only_hint)

        response = await client.call_tool(
            "rag_search_documents",
            {"question": "Quelle est la date limite de remise des offres ?", "method": "bm25", "top_k": 3},
        )
        assert not response.is_error, response.content
        data = response.structured_content
        assert data is not None and data["passages"]
        assert data["human_review_required"] is True
        print("Method:", data["method"])
        for passage in data["passages"]:
            print(f"{passage['source_id']}: {passage['document']} — page {passage['page']}")
        print("S3 MCP tool call: PASS")

await run_s3_mcp_demo()


In [ ]:
# Optional negative test: the tool schema should reject invalid top_k.
async def test_input_validation():
    async with Client(params) as client:
        response = await client.call_tool(
            "rag_search_documents",
            {"question": "Quelle est la date limite ?", "method": "bm25", "top_k": 999},
        )
        assert response.is_error
        print("Invalid top_k rejected: PASS")

await test_input_validation()


In [ ]:
# Optional: requires Ollama running with bge-m3 pulled on your local machine.
# Uncomment to demonstrate hybrid retrieval.
# async def run_hybrid():
#     async with Client(params) as client:
#         result = await client.call_tool("rag_search_documents", {
#             "question": "Quelle est la date limite de remise des offres ?",
#             "method": "hybrid", "top_k": 3,
#         })
#         assert not result.is_error, result.content
#         print(result.structured_content)
# await run_hybrid()
